In [ ]:
import sys
from rqvae.models.rqvae import RQVAE
from rqvae.datasets import TestDataset
import torch
from torch.utils.data import DataLoader
import numpy as np
def load_checkpoint_only(ckpt_path, map_location='cuda'):
    checkpoint = torch.load(ckpt_path, map_location=map_location, weights_only=False)
    return checkpoint

data_path = 'COLIEE/COLIEE_white_emb.npy'

dataset = TestDataset(data_path)
dataloader = DataLoader(dataset,
                        batch_size=1024,
                        shuffle=False,
                        num_workers=4
                        )

ckpt_path = 'pth for rqvae model'
params_dict = load_checkpoint_only(ckpt_path)
args = params_dict['args']
model = RQVAE(in_dim=512,
                  num_emb_list=args.num_emb_list,
                  e_dim=args.e_dim,
                  e_dim_list=args.e_dim_list,
                  layers=args.layers,
                  dropout_prob=args.dropout_prob,
                  norm_type=args.norm_type,
                  loss_type=args.loss_type,
                  quant_loss_weight=args.quant_loss_weight,
                  recon_loss_weight=args.recon_loss_weight,
                  kmeans_init=args.kmeans_init,
                  kmeans_iters=args.kmeans_iters,
                  margin_rate=args.margin_rate,
                  mu=args.mu,
                  n_clusters= args.n_clusters,
                  choose_method=args.choose_method
                  )
model.load_state_dict(params_dict['state_dict'])
model.eval()


codebook_emb = model.rq.get_codebook()
codebook_emb = np.stack([
    v.detach().cpu().numpy() for v in codebook_emb.values()
], axis=0)
print(codebook_emb.shape)
np.save('../../graph/COLIEE_codebook.npy', codebook_emb)


quanties = []
ids = []

for batch in dataloader:
    x_q, id = model.get_ids(batch)
    x_q = [t.detach().cpu().numpy() for t in x_q]
    quanties.append(x_q)
    ids.append(id)

layerwise_quanties = list(zip(*quanties))
data_quant = [np.concatenate(layer_list, axis=0) for layer_list in layerwise_quanties]

num_samples = data_quant[0].shape[0]
merged_per_sample = []

for i in range(num_samples):
    merged = [layer[i] for layer in data_quant]
    merged_per_sample.append(merged)


data_quant_obj = np.array(merged_per_sample, dtype=object)

np.save("../../graph/COLIEE_data_quant.npy", data_quant_obj, allow_pickle=True)

data_ids = torch.cat(ids)
data_ids = data_ids.numpy()
np.save("../../graph/COLIEE_data_ids.npy", data_ids)

print(args)

data_quant = np.load('../../graph/COLIEE_data_quant.npy', allow_pickle=True)
print(data_quant.shape)
for i in range(data_quant.shape[1]):
    print(len(data_quant[0][i]))

print(data_quant[0][0]==data_quant[1][0])
print(data_ids[4])
print(data_ids[985])

In [ ]:
charge_path = 'COLIEE/COLIEE_charge_white_emb.npy'
chargeset = TestDataset(charge_path)
chargeloader = DataLoader(chargeset,
                        batch_size=1024,
                        shuffle=False,
                        num_workers=4
                        )

quanties = []
ids = []

for batch in chargeloader:
    x_q, _ = model.get_ids(batch, 'test')
    x_q = [t.detach().cpu().numpy() for t in x_q]
    quanties.append(x_q)

layerwise_quanties = list(zip(*quanties))
charge_quant = [np.concatenate(layer_list, axis=0) for layer_list in layerwise_quanties]

num_samples = charge_quant[0].shape[0]
merged_per_sample = []

for i in range(num_samples):
    merged = [layer[i] for layer in charge_quant]
    merged_per_sample.append(merged)

charge_quant_obj = np.array(merged_per_sample, dtype=object)

np.save("../../graph/COLIEE_charge_quant.npy", charge_quant_obj, allow_pickle=True)